## Step 11 — merging of zones_2, constrained to zones_1 (initial zones) 
**# of cells in notebook:** 1

**Purpose:** With `zones_2`, we created unique features across the extent where each of those features has shared zone_1--density clutser--admin values. In theory, our workflow would be finished with the creation `zones_2` if each feature had a a population $\geq$ 100. In this step, we merge low population `zones_2` features with neighbors, but we constrain the exercise so that merging does not occur across `zones_1` values. If there are still low population features after step 11, we merge across `zones_1` in a subsequent step. 

**Input:**

- a geodatabase with `zones_2`
  
**Output:** `zones_3` layer, merge log with tracking fields

**Main logic:**

1. Copy `zones_2` to a working layer called `zones_3_work`, then initialize merge-tracking fields from the existing `concat` value. The script splits `concat` into its component parts: initial zone, admin/geoboundary value, and cluster value.
2. Repeatedly identify `zones_3_work` features whose `population` is below `100`, processing the lowest-population features first. After each merge, the script rebuilds the feature dictionary and polygon-neighbor relationships because geometry and adjacency have changed.
3. For each low-population target feature, find eligible shared-edge neighbors using a tiered priority system. The script never merges across the original/primary initial zone; within that constraint, it prefers neighbors with the same admin and same cluster, then same cluster/different admin, then same admin/different cluster, and finally different admin/different cluster.
4. If there is only one eligible neighbor in the best available tier, the target is merged into that neighbor directly. If there are multiple eligible neighbors in the best tier, the script evaluates each possible merge using Reock compactness and chooses the neighbor that produces the most compact merged geometry.
5. When a merge occurs, the selected neighbor is kept as the surviving feature and the low-population target is deleted. The surviving feature’s geometry, `population`, `block_count`, `conca`t, merge-tracking fields, and Reock audit fields are updated to reflect the absorbed feature.
6. The script records every merge in `zones_3_merge_log`, including the target, recipient neighbor, merge tier, population before/after, block counts, whether Reock was used, and the updated `concat` value.
7. When no remaining feature is below the population threshold, or no additional eligible merges can be made under the rules, the working layer is copied to the final output `zones_3`. The script then reports merge totals and lists any remaining features still < `100` population.

In [ ]:
import arcpy
import os
import traceback

# ------------------------------------------------------------
# Inputs / outputs
# ------------------------------------------------------------

gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"

zones_2 = os.path.join(gdb, "zones_2")
work_fc = os.path.join(gdb, "zones_3_work")
zones_3 = os.path.join(gdb, "zones_3")

neighbor_table = os.path.join(gdb, "zones_3_neighbors_work")
merge_log_table = os.path.join(gdb, "zones_3_merge_log")

population_field = "population"
block_count_field = "block_count"
concat_field = "concat"

# Tracking fields added to the working layer.
merge_z1_field = "merge_z1"
merge_admin_field = "merge_admin"
merge_cluster_field = "merge_cluster"
merge_count_field = "merge_count"

# New audit fields.
used_reock_field = "used_reock"              # 0/1: whether this final feature ever used Reock in its merge history
reock_merge_count_field = "reock_mrg_n"     # number of Reock-selected merges in this feature's history

# Stop condition.
population_threshold = 100

# ------------------------------------------------------------
# ArcPy environment
# ------------------------------------------------------------

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def field_exists(fc, field_name):
    return any(f.name.lower() == field_name.lower() for f in arcpy.ListFields(fc))


def add_text_field_if_missing(fc, field_name, length=500):
    if not field_exists(fc, field_name):
        arcpy.management.AddField(fc, field_name, "TEXT", field_length=length)
        print(f"Added text field: {field_name}")


def add_long_field_if_missing(fc, field_name):
    if not field_exists(fc, field_name):
        arcpy.management.AddField(fc, field_name, "LONG")
        print(f"Added long field: {field_name}")


def add_double_field_if_missing(fc, field_name):
    if not field_exists(fc, field_name):
        arcpy.management.AddField(fc, field_name, "DOUBLE")
        print(f"Added double field: {field_name}")


def parse_concat(value):
    """
    Expected format:
        initial_zones_1_sj_geoboundaries_cluster

    Example:
        18_1_2

    Returns:
        z1, admin, cluster as strings.
    """
    if value is None:
        return "", "", ""

    parts = str(value).split("_")

    if len(parts) < 3:
        return str(value), "", ""

    z1 = parts[0]
    admin = parts[1]
    cluster = "_".join(parts[2:])

    return z1, admin, cluster


def split_values(value):
    """
    Tracking fields may contain single values like:
        1

    or merged values like:
        1|2
    """
    if value is None:
        return set()

    value = str(value).strip()

    if value == "":
        return set()

    return set(v.strip() for v in value.split("|") if v.strip() != "")

def primary_value(value):
    """
    Returns the first/original value from a tracking field.

    Examples:
        "28"       -> "28"
        "28|11|10" -> "28"
        "2|1"      -> "2"
        None       -> ""
    """
    if value is None:
        return ""

    value = str(value).strip()

    if value == "":
        return ""

    return value.split("|")[0].strip()


def ordered_unique_values(values):
    """
    Preserve order while removing duplicates.
    """
    out = []
    seen = set()

    for v in values:
        if v is None:
            continue

        for part in str(v).split("|"):
            part = part.strip()
            if part == "":
                continue

            if part not in seen:
                out.append(part)
                seen.add(part)

    return "|".join(out)


def combine_tracking_values(primary_value, absorbed_value):
    """
    Keeps the surviving/recipient feature's values first,
    then appends absorbed feature values.
    """
    return ordered_unique_values([primary_value, absorbed_value])

def safe_float(value):
    if value is None:
        return 0.0
    return float(value)


def safe_int(value):
    if value is None:
        return 0
    return int(value)


def update_concat_from_tracking(z1, admin, cluster):
    return f"{z1}_{admin}_{cluster}"


def get_oid_field(fc):
    return arcpy.Describe(fc).OIDFieldName


def sql_text(value):
    """
    Escapes text values for SQL insertion into file geodatabase table.
    """
    if value is None:
        return ""
    return str(value).replace("'", "''")


# ------------------------------------------------------------
# Merge log table
# ------------------------------------------------------------

def create_merge_log_table():
    if arcpy.Exists(merge_log_table):
        arcpy.management.Delete(merge_log_table)

    print(f"Creating merge log table: {merge_log_table}")

    arcpy.management.CreateTable(gdb, os.path.basename(merge_log_table))

    add_long_field_if_missing(merge_log_table, "merge_id")
    add_long_field_if_missing(merge_log_table, "iteration")
    add_long_field_if_missing(merge_log_table, "target_oid")
    add_long_field_if_missing(merge_log_table, "neighbor_oid")
    add_long_field_if_missing(merge_log_table, "tier")
    add_long_field_if_missing(merge_log_table, "candidate_count")
    add_long_field_if_missing(merge_log_table, "used_reock")
    add_double_field_if_missing(merge_log_table, "reock_score")

    add_double_field_if_missing(merge_log_table, "target_pop")
    add_double_field_if_missing(merge_log_table, "neighbor_pop")
    add_double_field_if_missing(merge_log_table, "new_pop")

    add_long_field_if_missing(merge_log_table, "target_blocks")
    add_long_field_if_missing(merge_log_table, "neighbor_blocks")
    add_long_field_if_missing(merge_log_table, "new_blocks")

    add_text_field_if_missing(merge_log_table, "target_concat", length=500)
    add_text_field_if_missing(merge_log_table, "neighbor_concat", length=500)
    add_text_field_if_missing(merge_log_table, "new_concat", length=500)


def log_merge(
    merge_id,
    iteration,
    target_oid,
    neighbor_oid,
    tier,
    candidate_count,
    used_reock,
    reock_score,
    target_pop,
    neighbor_pop,
    new_pop,
    target_blocks,
    neighbor_blocks,
    new_blocks,
    target_concat,
    neighbor_concat,
    new_concat
):
    fields = [
        "merge_id",
        "iteration",
        "target_oid",
        "neighbor_oid",
        "tier",
        "candidate_count",
        "used_reock",
        "reock_score",
        "target_pop",
        "neighbor_pop",
        "new_pop",
        "target_blocks",
        "neighbor_blocks",
        "new_blocks",
        "target_concat",
        "neighbor_concat",
        "new_concat",
    ]

    with arcpy.da.InsertCursor(merge_log_table, fields) as icur:
        icur.insertRow([
            merge_id,
            iteration,
            target_oid,
            neighbor_oid,
            tier,
            candidate_count,
            used_reock,
            reock_score if reock_score is not None else None,
            target_pop,
            neighbor_pop,
            new_pop,
            target_blocks,
            neighbor_blocks,
            new_blocks,
            target_concat,
            neighbor_concat,
            new_concat,
        ])


# ------------------------------------------------------------
# Create working copy
# ------------------------------------------------------------

def create_initial_working_copy():
    if arcpy.Exists(work_fc):
        arcpy.management.Delete(work_fc)

    print("Copying zones_2 to zones_3_work...")
    arcpy.management.CopyFeatures(zones_2, work_fc)

    add_text_field_if_missing(work_fc, merge_z1_field, length=500)
    add_text_field_if_missing(work_fc, merge_admin_field, length=500)
    add_text_field_if_missing(work_fc, merge_cluster_field, length=500)
    add_long_field_if_missing(work_fc, merge_count_field)

    add_long_field_if_missing(work_fc, used_reock_field)
    add_long_field_if_missing(work_fc, reock_merge_count_field)

    if not field_exists(work_fc, population_field):
        raise RuntimeError(f"Required field not found: {population_field}")

    if not field_exists(work_fc, block_count_field):
        raise RuntimeError(f"Required field not found: {block_count_field}")

    if not field_exists(work_fc, concat_field):
        raise RuntimeError(f"Required field not found: {concat_field}")

    fields = [
        concat_field,
        merge_z1_field,
        merge_admin_field,
        merge_cluster_field,
        merge_count_field,
        used_reock_field,
        reock_merge_count_field,
    ]

    print("Initializing merge tracking fields from concat...")
    with arcpy.da.UpdateCursor(work_fc, fields) as cur:
        for row in cur:
            concat_value = row[0]
            z1, admin, cluster = parse_concat(concat_value)

            row[1] = z1
            row[2] = admin
            row[3] = cluster
            row[4] = 1
            row[5] = 0
            row[6] = 0

            cur.updateRow(row)

    print("Working copy ready.")


# ------------------------------------------------------------
# Read current working feature class state
# ------------------------------------------------------------

def build_feature_dict(fc):
    oid_field = get_oid_field(fc)

    fields = [
        oid_field,
        "SHAPE@",
        population_field,
        block_count_field,
        concat_field,
        merge_z1_field,
        merge_admin_field,
        merge_cluster_field,
        merge_count_field,
        used_reock_field,
        reock_merge_count_field,
    ]

    data = {}

    with arcpy.da.SearchCursor(fc, fields) as cur:
        for row in cur:
            oid = row[0]

            data[oid] = {
                "oid": oid,
                "geometry": row[1],
                "population": safe_float(row[2]),
                "block_count": safe_int(row[3]),
                "concat": row[4],
                "z1": row[5],
                "admin": row[6],
                "cluster": row[7],
                "merge_count": safe_int(row[8]) if row[8] is not None else 1,
                "used_reock": safe_int(row[9]),
                "reock_merge_count": safe_int(row[10]),
            }

    return data


# ------------------------------------------------------------
# Build edge-neighbor dictionary
# ------------------------------------------------------------

def build_edge_neighbors(fc):
    """
    Rebuilds polygon neighbor table and returns dictionary:

        neighbors[src_oid] = [
            {"nbr_oid": ..., "length": ...},
            ...
        ]

    Only keeps shared-edge neighbors, where LENGTH > 0.
    """

    if arcpy.Exists(neighbor_table):
        arcpy.management.Delete(neighbor_table)

    oid_field = get_oid_field(fc)

    print("Recomputing edge neighbors...")

    arcpy.analysis.PolygonNeighbors(
        in_features=fc,
        out_table=neighbor_table,
        in_fields=oid_field,
        area_overlap="NO_AREA_OVERLAP",
        both_sides="BOTH_SIDES",
        cluster_tolerance=None,
        out_linear_units="METERS",
        out_area_units="SQUARE_METERS"
    )

    fields = [f.name for f in arcpy.ListFields(neighbor_table)]

    src_field = None
    nbr_field = None
    length_field = None

    for f in fields:
        fl = f.lower()

        if fl.startswith("src_") and fl.endswith(oid_field.lower()):
            src_field = f

        if fl.startswith("nbr_") and fl.endswith(oid_field.lower()):
            nbr_field = f

        if fl == "length":
            length_field = f

    if src_field is None and "src_OBJECTID" in fields:
        src_field = "src_OBJECTID"

    if nbr_field is None and "nbr_OBJECTID" in fields:
        nbr_field = "nbr_OBJECTID"

    if length_field is None and "LENGTH" in fields:
        length_field = "LENGTH"

    if src_field is None or nbr_field is None or length_field is None:
        raise RuntimeError(
            "Could not identify src/nbr/LENGTH fields in PolygonNeighbors output. "
            f"Fields found: {fields}"
        )

    neighbors = {}

    with arcpy.da.SearchCursor(neighbor_table, [src_field, nbr_field, length_field]) as cur:
        for src_oid, nbr_oid, shared_len in cur:
            if src_oid is None or nbr_oid is None:
                continue

            if shared_len is None:
                continue

            # Exclude vertex-only contacts.
            if float(shared_len) <= 0:
                continue

            neighbors.setdefault(src_oid, []).append({
                "nbr_oid": nbr_oid,
                "length": float(shared_len)
            })

    return neighbors

# ------------------------------------------------------------
# Candidate priority logic
# ------------------------------------------------------------

def classify_candidate(target, neighbor):
    """
    Priority structure:

    Tier 1:
        same primary zone, same primary admin, same primary cluster

    Tier 2:
        same primary zone, different primary admin, same primary cluster

    Tier 3:
        same primary zone, same primary admin, different primary cluster

    Tier 4:
        same primary zone, different primary admin, different primary cluster

    Important:
        Tracking fields may store merge histories like:

            z1      = 28
            admin   = 1
            cluster = 2|1

        For merge eligibility and candidate ranking, we interpret this
        by the first/original value only:

            28_1_2|1 behaves like 28_1_2

        The full pipe-delimited value is still preserved in the output
        concat field as an audit trail.
    """

    target_z1 = primary_value(target["z1"])
    nbr_z1 = primary_value(neighbor["z1"])

    target_admin = primary_value(target["admin"])
    nbr_admin = primary_value(neighbor["admin"])

    target_cluster = primary_value(target["cluster"])
    nbr_cluster = primary_value(neighbor["cluster"])

    # Never merge across primary/original zone.
    if target_z1 != nbr_z1:
        return None

    same_admin = target_admin == nbr_admin
    same_cluster = target_cluster == nbr_cluster

    if same_admin and same_cluster:
        return 1

    if not same_admin and same_cluster:
        return 2

    if same_admin and not same_cluster:
        return 3

    if not same_admin and not same_cluster:
        return 4

    return None

# ------------------------------------------------------------
# Reock compactness
# ------------------------------------------------------------

def reock_compactness_for_geometry(geometry, spatial_reference):
    """
    Reock compactness:
        polygon area / area of minimum bounding circle
    """

    if geometry is None:
        return -1

    geom_area = float(geometry.area)

    if geom_area <= 0:
        return -1

    temp_fc = r"in_memory\candidate_merge_geom"
    temp_circle = r"in_memory\candidate_merge_circle"

    for temp in [temp_fc, temp_circle]:
        if arcpy.Exists(temp):
            arcpy.management.Delete(temp)

    arcpy.management.CreateFeatureclass(
        out_path="in_memory",
        out_name="candidate_merge_geom",
        geometry_type="POLYGON",
        spatial_reference=spatial_reference
    )

    with arcpy.da.InsertCursor(temp_fc, ["SHAPE@"]) as icur:
        icur.insertRow([geometry])

    arcpy.management.MinimumBoundingGeometry(
        in_features=temp_fc,
        out_feature_class=temp_circle,
        geometry_type="CIRCLE",
        group_option="ALL"
    )

    circle_area = None

    with arcpy.da.SearchCursor(temp_circle, ["SHAPE@AREA"]) as cur:
        for row in cur:
            circle_area = float(row[0])
            break

    for temp in [temp_fc, temp_circle]:
        if arcpy.Exists(temp):
            arcpy.management.Delete(temp)

    if circle_area is None or circle_area <= 0:
        return -1

    return geom_area / circle_area


def choose_best_neighbor_by_reock(target_oid, candidate_oids, feature_dict, spatial_reference):
    """
    If there is one candidate, return it directly and do not use Reock.
    If multiple, union target geometry with each candidate geometry and choose
    the candidate with highest Reock compactness.
    """

    if len(candidate_oids) == 0:
        return None, None, 0

    if len(candidate_oids) == 1:
        return candidate_oids[0], None, 0

    target_geom = feature_dict[target_oid]["geometry"]

    best_oid = None
    best_score = -1

    for nbr_oid in candidate_oids:
        nbr_geom = feature_dict[nbr_oid]["geometry"]

        try:
            merged_geom = target_geom.union(nbr_geom)
            score = reock_compactness_for_geometry(merged_geom, spatial_reference)

            if score > best_score:
                best_score = score
                best_oid = nbr_oid

        except Exception:
            print(
                f"WARNING: Could not evaluate Reock compactness for "
                f"target {target_oid}, neighbor {nbr_oid}"
            )
            traceback.print_exc()

    return best_oid, best_score, 1


# ------------------------------------------------------------
# Merge operation
# ------------------------------------------------------------

def merge_target_into_neighbor(fc, target_oid, nbr_oid, feature_dict, used_reock_for_this_merge):
    """
    Keeps neighbor / recipient feature.
    Deletes target / low-population feature.

    This is important because the selected neighbor is the recipient.
    Tracking values are ordered as:
        recipient first, absorbed target second

    Updates:
        geometry
        population
        block_count
        concat
        merge_z1
        merge_admin
        merge_cluster
        merge_count
        used_reock
        reock_mrg_n
    """

    oid_field = get_oid_field(fc)

    target = feature_dict[target_oid]  # low-population feature being absorbed
    nbr = feature_dict[nbr_oid]        # selected recipient / surviving feature

    merged_geom = nbr["geometry"].union(target["geometry"])

    merged_population = (
        safe_float(nbr["population"]) +
        safe_float(target["population"])
    )

    merged_block_count = (
        safe_int(nbr["block_count"]) +
        safe_int(target["block_count"])
    )

    # Recipient-first ordering
    merged_z1 = combine_tracking_values(nbr["z1"], target["z1"])
    merged_admin = combine_tracking_values(nbr["admin"], target["admin"])
    merged_cluster = combine_tracking_values(nbr["cluster"], target["cluster"])

    merged_count = (
        safe_int(nbr["merge_count"]) +
        safe_int(target["merge_count"])
    )

    merged_used_reock = max(
        safe_int(nbr["used_reock"]),
        safe_int(target["used_reock"]),
        safe_int(used_reock_for_this_merge)
    )

    merged_reock_merge_count = (
        safe_int(nbr["reock_merge_count"]) +
        safe_int(target["reock_merge_count"]) +
        safe_int(used_reock_for_this_merge)
    )

    merged_concat = update_concat_from_tracking(
        merged_z1,
        merged_admin,
        merged_cluster
    )

    update_fields = [
        oid_field,
        "SHAPE@",
        population_field,
        block_count_field,
        concat_field,
        merge_z1_field,
        merge_admin_field,
        merge_cluster_field,
        merge_count_field,
        used_reock_field,
        reock_merge_count_field,
    ]

    # Update the neighbor / recipient feature
    where_nbr = f"{arcpy.AddFieldDelimiters(fc, oid_field)} = {nbr_oid}"

    with arcpy.da.UpdateCursor(fc, update_fields, where_nbr) as ucur:
        updated = False

        for row in ucur:
            row[1] = merged_geom
            row[2] = merged_population
            row[3] = merged_block_count
            row[4] = merged_concat
            row[5] = merged_z1
            row[6] = merged_admin
            row[7] = merged_cluster
            row[8] = merged_count
            row[9] = merged_used_reock
            row[10] = merged_reock_merge_count

            ucur.updateRow(row)
            updated = True

        if not updated:
            raise RuntimeError(f"Could not update recipient/neighbor OBJECTID {nbr_oid}")

    # Delete the target / absorbed low-population feature
    where_target = f"{arcpy.AddFieldDelimiters(fc, oid_field)} = {target_oid}"

    with arcpy.da.UpdateCursor(fc, [oid_field], where_target) as dcur:
        deleted = False

        for row in dcur:
            dcur.deleteRow()
            deleted = True

        if not deleted:
            raise RuntimeError(f"Could not delete absorbed target OBJECTID {target_oid}")

    return (
        merged_population,
        merged_block_count,
        merged_concat,
        merged_used_reock,
        merged_reock_merge_count
    )

# ------------------------------------------------------------
# Reporting
# ------------------------------------------------------------

def summarize_remaining_low_pop(fc):
    oid_field = get_oid_field(fc)

    low = []

    with arcpy.da.SearchCursor(
        fc,
        [oid_field, population_field, block_count_field, concat_field]
    ) as cur:
        for oid, pop, block_count, concat in cur:
            pop = safe_float(pop)

            if pop < population_threshold:
                low.append((oid, pop, block_count, concat))

    low.sort(key=lambda x: x[1])
    return low


def summarize_reock_from_log():
    total_merges = 0
    single_candidate_merges = 0
    reock_selected_merges = 0

    if not arcpy.Exists(merge_log_table):
        return total_merges, single_candidate_merges, reock_selected_merges

    with arcpy.da.SearchCursor(merge_log_table, ["used_reock"]) as cur:
        for row in cur:
            total_merges += 1

            if safe_int(row[0]) == 1:
                reock_selected_merges += 1
            else:
                single_candidate_merges += 1

    return total_merges, single_candidate_merges, reock_selected_merges


# ------------------------------------------------------------
# Main iterative merge process
# ------------------------------------------------------------

def run_iterative_merges():
    create_initial_working_copy()
    create_merge_log_table()

    spatial_reference = arcpy.Describe(work_fc).spatialReference

    iteration = 0
    total_merges = 0
    single_candidate_merges = 0
    reock_selected_merges = 0

    blocked_this_pass = set()

    while True:
        iteration += 1

        feature_dict = build_feature_dict(work_fc)
        neighbors = build_edge_neighbors(work_fc)

        low_pop_oids = [
            oid for oid, data in feature_dict.items()
            if safe_float(data["population"]) < population_threshold
        ]

        # Important:
        # This sorts by population, not by OBJECTID.
        # That is why OBJECTIDs may appear out of numeric order in the printout.
        low_pop_oids.sort(key=lambda oid: safe_float(feature_dict[oid]["population"]))

        if not low_pop_oids:
            print("\nNo features remain below population threshold.")
            break

        print("\n" + "-" * 70)
        print(f"Iteration {iteration}")
        print(f"Features below {population_threshold}: {len(low_pop_oids)}")
        print(f"Total features currently: {len(feature_dict)}")
        print("-" * 70)

        merge_performed = False

        for target_oid in low_pop_oids:
            if target_oid in blocked_this_pass:
                continue

            target = feature_dict[target_oid]
            nbr_infos = neighbors.get(target_oid, [])

            if not nbr_infos:
                print(
                    f"OBJECTID {target_oid}: "
                    f"population={target['population']:.2f}; "
                    "no edge neighbors."
                )
                blocked_this_pass.add(target_oid)
                continue

            tier_candidates = {
                1: [],
                2: [],
                3: [],
                4: [],
            }

            for nbr_info in nbr_infos:
                nbr_oid = nbr_info["nbr_oid"]

                if nbr_oid not in feature_dict:
                    continue

                nbr = feature_dict[nbr_oid]
                tier = classify_candidate(target, nbr)

                if tier is not None:
                    tier_candidates[tier].append(nbr_oid)

            chosen_tier = None
            candidate_oids = []

            for tier in [1, 2, 3, 4]:
                if tier_candidates[tier]:
                    chosen_tier = tier
                    candidate_oids = tier_candidates[tier]
                    break

            if not candidate_oids:
                print(
                    f"OBJECTID {target_oid}: "
                    f"population={target['population']:.2f}; "
                    "has edge neighbors but no eligible same-initial-zone candidate."
                )
                blocked_this_pass.add(target_oid)
                continue

            candidate_count = len(candidate_oids)

            best_nbr_oid, reock_score, used_reock_for_this_merge = choose_best_neighbor_by_reock(
                target_oid,
                candidate_oids,
                feature_dict,
                spatial_reference
            )

            if best_nbr_oid is None:
                print(f"OBJECTID {target_oid}: no valid Reock-selected neighbor.")
                blocked_this_pass.add(target_oid)
                continue

            old_pop = target["population"]
            old_block_count = target["block_count"]
            old_concat = target["concat"]

            nbr_pop = feature_dict[best_nbr_oid]["population"]
            nbr_block_count = feature_dict[best_nbr_oid]["block_count"]
            nbr_concat = feature_dict[best_nbr_oid]["concat"]

            (
                new_pop,
                new_block_count,
                new_concat,
                merged_used_reock,
                merged_reock_merge_count
            ) = merge_target_into_neighbor(
                work_fc,
                target_oid,
                best_nbr_oid,
                feature_dict,
                used_reock_for_this_merge
            )

            total_merges += 1

            if used_reock_for_this_merge == 1:
                reock_selected_merges += 1
            else:
                single_candidate_merges += 1

            log_merge(
                merge_id=total_merges,
                iteration=iteration,
                target_oid=target_oid,
                neighbor_oid=best_nbr_oid,
                tier=chosen_tier,
                candidate_count=candidate_count,
                used_reock=used_reock_for_this_merge,
                reock_score=reock_score,
                target_pop=old_pop,
                neighbor_pop=nbr_pop,
                new_pop=new_pop,
                target_blocks=old_block_count,
                neighbor_blocks=nbr_block_count,
                new_blocks=new_block_count,
                target_concat=old_concat,
                neighbor_concat=nbr_concat,
                new_concat=new_concat
            )

            merge_performed = True
            blocked_this_pass = set()

            if used_reock_for_this_merge == 1:
                reock_msg = f"Reock-selected merge; Reock={reock_score:.4f}; candidates={candidate_count}"
            else:
                reock_msg = "single eligible neighbor; Reock not used"

            print(
                f"MERGE {total_merges}: "
                f"target OBJECTID {target_oid} "
                f"pop={old_pop:.2f}, blocks={old_block_count} "
                f"absorbed into recipient OBJECTID {best_nbr_oid} "
                f"pop={nbr_pop:.2f}, blocks={nbr_block_count} "
                f"-> new pop={new_pop:.2f}, new blocks={new_block_count}; "
                f"tier={chosen_tier}; {reock_msg}; "
                f"new concat={new_concat}"
            )

            # After one merge, geometry and neighbor relationships changed.
            # Recompute from scratch before the next merge.
            break

        if not merge_performed:
            print("\nNo further eligible merges can be performed under the current rules.")
            break

    # Save final result.
    if arcpy.Exists(zones_3):
        arcpy.management.Delete(zones_3)

    print("\nWriting final output zones_3...")
    arcpy.management.CopyFeatures(work_fc, zones_3)

    remaining = summarize_remaining_low_pop(zones_3)

    # Cross-check from the log table.
    (
        log_total_merges,
        log_single_candidate_merges,
        log_reock_selected_merges
    ) = summarize_reock_from_log()

    print("\n" + "=" * 70)
    print("DONE")
    print(f"Total merges performed: {total_merges}")
    print(f"Single-candidate merges: {single_candidate_merges}")
    print(f"Reock-selected merges: {reock_selected_merges}")
    print(f"Merge log table: {merge_log_table}")
    print(f"Final output: {zones_3}")
    print(f"Remaining features below {population_threshold}: {len(remaining)}")
    print("=" * 70)

    print("\nLog table cross-check:")
    print(f"  Total log records: {log_total_merges}")
    print(f"  Single-candidate log records: {log_single_candidate_merges}")
    print(f"  Reock-selected log records: {log_reock_selected_merges}")

    if remaining:
        print("\nRemaining low-population features:")
        for oid, pop, block_count, concat in remaining[:100]:
            print(
                f"  OBJECTID {oid}: "
                f"population={pop:.2f}, "
                f"block_count={block_count}, "
                f"concat={concat}"
            )

        if len(remaining) > 100:
            print(f"  ... plus {len(remaining) - 100} more")


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

run_iterative_merges()